In [ ]:
from pathlib import Path

import numpy as np
import xarray as xr
from scipy.ndimage import uniform_filter


# ============================================================
# Paths
# ============================================================

RAW_DATA_DIR = Path("../../data/interim/SWOT_L3_BOB_Dataset/raw_data")
LABEL_DIR = Path("../../data/interim/SWOT_L3_BOB_Dataset/labels")
OUT_DIR = Path("../../data/interim/SWOT_L3_BOB_Dataset/processed_data")


# ============================================================
# Variables
# ============================================================

BASE_VAR = "ssha_unfiltered"
QUALITY_VAR = "quality_flag"

MASK_VAR = "iw_bbox_mask"
HIGHPASS_VAR = "rolling_mean_highpass"


# ============================================================
# Filter parameters
# Same rolling setup as your image-generation code
# ============================================================

SIGMA = 8
ROLLING_WINDOW = int(SIGMA) * 2

PIXEL_SIZE_KM = 2.0
MAX_VALID_VALUE = 10.0

GOOD_QUALITY_VALUES = np.array([0, 5, 30, 50])


# ============================================================
# Helper functions
# ============================================================

def select_labeled_2d_da(da):
    """
    Select the same 2D slice that was used for image generation.

    Your original plotting code did:
        ds_plot = ds.isel(file=0) if "file" in ds.dims else ds
        arr = np.squeeze(da.values)

    This function mirrors that behavior, but keeps track of selectors
    so the processed variables can be written back with the same dims
    as the original NetCDF variable.
    """
    selectors = {}

    if "file" in da.dims:
        selectors["file"] = 0

    # Match np.squeeze behavior for singleton dimensions.
    for dim in da.dims:
        if dim in selectors:
            continue
        if da.sizes[dim] == 1:
            selectors[dim] = 0

    da2 = da.isel(selectors).squeeze(drop=True)

    if da2.ndim != 2:
        raise ValueError(
            f"Expected 2D array after selecting/squeezing, got "
            f"shape={da2.shape}, dims={da2.dims}. "
            f"Add any needed dimension selector to select_labeled_2d_da()."
        )

    return da2, selectors


def apply_quality_flag_mask(ssha, quality_flag):
    """
    Keep only quality_flag values 0, 5, 30, and 50.
    Everything else becomes NaN.
    Also masks values > MAX_VALID_VALUE, matching your image code.
    """
    ssha = ssha.astype(np.float32).copy()

    good_quality = np.isin(quality_flag, GOOD_QUALITY_VALUES)
    ssha[~good_quality] = np.nan

    # Same bad-value rule used before saving JPGs.
    ssha = np.where(ssha <= MAX_VALID_VALUE, ssha, np.nan).astype(np.float32)

    return ssha


def nan_uniform_filter(arr, window):
    """
    Rolling/uniform mean filter that handles NaNs.
    """
    arr = arr.astype(np.float32)

    valid = np.isfinite(arr)
    arr_filled = np.where(valid, arr, 0.0).astype(np.float32)
    weights = valid.astype(np.float32)

    smooth_sum = uniform_filter(
        arr_filled,
        size=window,
        mode="nearest",
    )

    weight_sum = uniform_filter(
        weights,
        size=window,
        mode="nearest",
    )

    smooth = smooth_sum / np.maximum(weight_sum, 1e-6)
    smooth[weight_sum <= 1e-6] = np.nan

    return smooth.astype(np.float32)


def rolling_highpass(arr, window):
    """
    High-pass = original - rolling/uniform low-pass.
    """
    arr = arr.astype(np.float32)

    valid = np.isfinite(arr)
    smooth = nan_uniform_filter(arr, window=window)

    highpass = arr - smooth
    highpass[~valid] = np.nan

    return highpass.astype(np.float32)


# ============================================================
# YOLO bbox -> original NetCDF grid mask
# ============================================================

def image_shape_from_original_shape(original_shape):
    """
    Determine the saved JPG pixel shape from the original 2D array shape.

    Your image pipeline was:

        if arr.shape[1] > arr.shape[0]:
            arr = arr.T

        arr = np.fliplr(arr)

    np.fliplr does not change shape.
    """
    h, w = original_shape

    if w > h:
        return (w, h)

    return (h, w)


def yolo_labels_to_image_mask(label_path, image_shape):
    """
    Convert YOLO bbox labels into a binary mask in saved-image orientation.

    YOLO bbox format expected:
        class_id x_center y_center width height

    Coordinates are normalized to [0, 1].
    """
    image_h, image_w = image_shape
    mask_img = np.zeros((image_h, image_w), dtype=np.uint8)

    if not label_path.exists():
        return mask_img, 0

    box_count = 0

    with label_path.open("r") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line or line.startswith("#"):
                continue

            parts = line.split()

            if len(parts) < 5:
                print(f"Skipping malformed label line {line_number} in {label_path}: {line}")
                continue

            try:
                class_id = int(float(parts[0]))
                x_center = float(parts[1])
                y_center = float(parts[2])
                box_w = float(parts[3])
                box_h = float(parts[4])
            except ValueError:
                print(f"Skipping non-numeric label line {line_number} in {label_path}: {line}")
                continue

            vals = np.array([x_center, y_center, box_w, box_h], dtype=np.float64)
            if not np.all(np.isfinite(vals)):
                print(f"Skipping non-finite label line {line_number} in {label_path}: {line}")
                continue

            # Convert normalized YOLO coordinates to image pixel bounds.
            x0 = int(np.floor((x_center - box_w / 2.0) * image_w))
            x1 = int(np.ceil((x_center + box_w / 2.0) * image_w))
            y0 = int(np.floor((y_center - box_h / 2.0) * image_h))
            y1 = int(np.ceil((y_center + box_h / 2.0) * image_h))

            # Clip to image bounds.
            x0 = max(0, min(image_w, x0))
            x1 = max(0, min(image_w, x1))
            y0 = max(0, min(image_h, y0))
            y1 = max(0, min(image_h, y1))

            if x1 <= x0 or y1 <= y0:
                print(f"Skipping empty/clipped bbox line {line_number} in {label_path}: {line}")
                continue

            # Binary detection mask:
            #   0 = outside any IW bbox
            #   1 = inside at least one IW bbox
            mask_img[y0:y1, x0:x1] = 1
            box_count += 1

    return mask_img, box_count


def image_mask_to_original_grid(mask_img, original_shape):
    """
    Reverse the exact image transform used before labeling.

    Forward image transform:
        original_arr
        -> optional transpose if width > height
        -> np.fliplr
        -> saved image / YOLO label coordinates

    Inverse transform:
        YOLO image mask
        -> undo np.fliplr
        -> undo optional transpose
        -> original NetCDF grid mask
    """
    original_h, original_w = original_shape
    was_transposed = original_w > original_h

    # Undo left-right mirror.
    mask = np.fliplr(mask_img)

    # Undo optional transpose.
    if was_transposed:
        mask = mask.T

    if mask.shape != original_shape:
        raise ValueError(
            f"Mask alignment failed. Expected original shape {original_shape}, "
            f"got {mask.shape}."
        )

    return mask.astype(np.uint8)


def make_bbox_mask_for_nc_grid(label_path, original_shape):
    """
    Build a YOLO bbox mask aligned to the original NetCDF grid.
    """
    image_shape = image_shape_from_original_shape(original_shape)
    mask_img, box_count = yolo_labels_to_image_mask(label_path, image_shape)
    mask_grid = image_mask_to_original_grid(mask_img, original_shape)

    return mask_grid, box_count, image_shape


# ============================================================
# Write processed variables with same dims as BASE_VAR
# ============================================================

def make_full_sized_da(base_da, arr2d, spatial_dims, selectors, fill_value, dtype, attrs):
    """
    Create a DataArray with the same dims as base_da.

    The processed 2D array is inserted into the selected slice, usually file=0.
    Other slices are filled with fill_value.

    For ordinary 2D variables, this just writes the 2D array directly.
    """
    full_shape = tuple(base_da.sizes[dim] for dim in base_da.dims)
    full = np.full(full_shape, fill_value, dtype=dtype)

    indexer = []

    for dim in base_da.dims:
        if dim in spatial_dims:
            indexer.append(slice(None))
        elif dim in selectors:
            indexer.append(selectors[dim])
        else:
            raise ValueError(
                f"Dimension {dim!r} is neither spatial nor selected. "
                f"base_da dims={base_da.dims}, spatial_dims={spatial_dims}, "
                f"selectors={selectors}"
            )

    full[tuple(indexer)] = arr2d.astype(dtype)

    coords = {
        dim: base_da.coords[dim]
        for dim in base_da.dims
        if dim in base_da.coords
    }

    return xr.DataArray(
        full,
        dims=base_da.dims,
        coords=coords,
        attrs=attrs,
    )


# ============================================================
# Main processing loop
# ============================================================

def process_one_file(nc_file):
    label_path = LABEL_DIR / f"{nc_file.stem}.txt"
    out_nc = OUT_DIR / nc_file.name

    with xr.open_dataset(nc_file) as ds:
        ds_out = ds.load()

    if BASE_VAR not in ds_out:
        raise KeyError(f"Missing base variable {BASE_VAR!r} in {nc_file}")

    if QUALITY_VAR not in ds_out:
        raise KeyError(f"Missing quality variable {QUALITY_VAR!r} in {nc_file}")

    base_da = ds_out[BASE_VAR]
    quality_da = ds_out[QUALITY_VAR]

    base_2d_da, base_selectors = select_labeled_2d_da(base_da)
    quality_2d_da, quality_selectors = select_labeled_2d_da(quality_da)

    if base_2d_da.shape != quality_2d_da.shape:
        raise ValueError(
            f"Shape mismatch in {nc_file}: "
            f"{BASE_VAR} shape={base_2d_da.shape}, "
            f"{QUALITY_VAR} shape={quality_2d_da.shape}"
        )

    if base_2d_da.dims != quality_2d_da.dims:
        print(
            f"Warning: spatial dims differ in {nc_file}: "
            f"{BASE_VAR} dims={base_2d_da.dims}, "
            f"{QUALITY_VAR} dims={quality_2d_da.dims}"
        )

    original_shape = base_2d_da.shape
    spatial_dims = base_2d_da.dims

    ssha = base_2d_da.values.astype(np.float32)
    quality_flag = quality_2d_da.values

    ssha_for_filtering = apply_quality_flag_mask(ssha, quality_flag)

    highpass_2d = rolling_highpass(
        ssha_for_filtering,
        window=ROLLING_WINDOW,
    )

    bbox_mask_2d, box_count, image_shape = make_bbox_mask_for_nc_grid(
        label_path=label_path,
        original_shape=original_shape,
    )

    mask_attrs = {
        "long_name": "IW YOLO bounding-box detection mask",
        "description": (
            "Binary mask made from YOLO bbox labels. 1 indicates pixels inside "
            "at least one IW bounding box; 0 indicates outside all IW bounding boxes."
        ),
        "flag_values": np.array([0, 1], dtype=np.uint8),
        "flag_meanings": "outside_iw_bbox inside_iw_bbox",
        "source_label_file": str(label_path),
        "label_file_found": str(label_path.exists()),
        "yolo_format": "class_id x_center y_center width height, normalized to saved image size",
        "alignment_note": (
            "Labels were drawn on images created from the NetCDF grid by optional "
            "transpose when width > height, followed by np.fliplr. This mask reverses "
            "np.fliplr first, then reverses the optional transpose."
        ),
        "original_grid_shape": str(original_shape),
        "label_image_shape": str(image_shape),
        "bbox_count": int(box_count),
    }

    highpass_attrs = {
        "long_name": f"{BASE_VAR} rolling-mean high-pass",
        "description": (
            f"High-pass field computed as {BASE_VAR} minus NaN-aware rolling mean. "
            f"Quality flag masking and value > {MAX_VALID_VALUE} masking are applied "
            "before filtering, matching the image-generation pipeline."
        ),
        "source_variable": BASE_VAR,
        "quality_variable": QUALITY_VAR,
        "good_quality_values": ", ".join(str(v) for v in GOOD_QUALITY_VALUES.tolist()),
        "rolling_window_pixels": int(ROLLING_WINDOW),
        "rolling_window_km": float(ROLLING_WINDOW * PIXEL_SIZE_KM),
        "pixel_size_km_assumed": float(PIXEL_SIZE_KM),
        "bad_value_rule": f"{BASE_VAR} > {MAX_VALID_VALUE} set to NaN before filtering",
    }

    ds_out[MASK_VAR] = make_full_sized_da(
        base_da=base_da,
        arr2d=bbox_mask_2d,
        spatial_dims=spatial_dims,
        selectors=base_selectors,
        fill_value=0,
        dtype=np.uint8,
        attrs=mask_attrs,
    )

    ds_out[HIGHPASS_VAR] = make_full_sized_da(
        base_da=base_da,
        arr2d=highpass_2d,
        spatial_dims=spatial_dims,
        selectors=base_selectors,
        fill_value=np.nan,
        dtype=np.float32,
        attrs=highpass_attrs,
    )

    ds_out.attrs["processed_by"] = "add_yolo_masks_and_highpass_to_nc.py"
    ds_out.attrs["processed_data_note"] = (
        f"Added {MASK_VAR} and {HIGHPASS_VAR}. "
        "YOLO bbox mask was inverse-mapped from saved-image coordinates back to NetCDF grid coordinates."
    )

    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # Write atomically-ish: create temp first, then replace final file.
    tmp_nc = out_nc.with_suffix(out_nc.suffix + ".tmp")

    if tmp_nc.exists():
        tmp_nc.unlink()

    ds_out.to_netcdf(tmp_nc)

    if out_nc.exists():
        out_nc.unlink()

    tmp_nc.replace(out_nc)

    mask_pixels = int(np.sum(bbox_mask_2d == 1))

    return {
        "nc_file": nc_file,
        "label_path": label_path,
        "out_nc": out_nc,
        "label_found": label_path.exists(),
        "box_count": box_count,
        "mask_pixels": mask_pixels,
        "original_shape": original_shape,
        "image_shape": image_shape,
    }


def main():
    nc_files = sorted(RAW_DATA_DIR.glob("*.nc"))

    print(f"Looking for NetCDF files in: {RAW_DATA_DIR}")
    print(f"Looking for YOLO labels in:   {LABEL_DIR}")
    print(f"Saving processed NetCDFs to:  {OUT_DIR}")
    print(f"Found NetCDF files: {len(nc_files)}")
    print(f"Rolling window: {ROLLING_WINDOW} px = {ROLLING_WINDOW * PIXEL_SIZE_KM} km")

    if not nc_files:
        raise FileNotFoundError(f"No .nc files found in {RAW_DATA_DIR}")

    processed = 0
    failed = 0
    missing_labels = []
    bad_items = []

    for nc_file in nc_files:
        print(f"\nProcessing: {nc_file}")

        try:
            result = process_one_file(nc_file)

            processed += 1

            print(f"Saved: {result['out_nc']}")
            print(f"Original grid shape: {result['original_shape']}")
            print(f"YOLO image shape:    {result['image_shape']}")
            print(f"Label found:         {result['label_found']}")
            print(f"YOLO boxes:          {result['box_count']}")
            print(f"Mask pixels:         {result['mask_pixels']}")

            if not result["label_found"]:
                missing_labels.append(str(result["label_path"]))

        except Exception as e:
            failed += 1
            bad_items.append(f"{nc_file} | {type(e).__name__}: {e}")
            print(f"FAILED: {nc_file}")
            print(f"Error: {type(e).__name__}: {e}")

    print("\nDone.")
    print(f"Processed files: {processed}")
    print(f"Failed files:    {failed}")

    if missing_labels:
        print("\nFiles with no matching YOLO label txt; all-zero masks were written:")
        for item in missing_labels:
            print(item)

    if bad_items:
        print("\nFailed items:")
        for item in bad_items:
            print(item)


if __name__ == "__main__":
    main()

In [ ]:
from pathlib import Path
import random

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt


# ============================================================
# Notebook settings
# ============================================================

%matplotlib inline


# ============================================================
# Paths
# ============================================================

PROCESSED_DIR = Path("../../data/interim/SWOT_L3_BOB_Dataset/processed_data")


# ============================================================
# Variables
# ============================================================

BASE_VAR = "ssha_unfiltered"
HIGHPASS_VAR = "rolling_mean_highpass"
MASK_VAR = "iw_bbox_mask"


# ============================================================
# Helpers
# ============================================================

def get_2d_da(ds, var_name):
    """
    Get a 2D DataArray from the processed NetCDF.

    No transpose.
    No flip.
    """
    if var_name not in ds:
        raise KeyError(f"Missing variable: {var_name}")

    da = ds[var_name]

    if "file" in da.dims:
        da = da.isel(file=0)

    da = da.squeeze(drop=True)

    if da.ndim != 2:
        raise ValueError(
            f"Expected {var_name} to be 2D after squeeze, "
            f"but got shape={da.shape}, dims={da.dims}"
        )

    return da


def robust_limits(arr, lower=2.5, upper=97.5):
    """
    Percentile color limits ignoring NaNs.
    """
    arr = np.asarray(arr)

    if not np.isfinite(arr).any():
        return None, None

    vmin, vmax = np.nanpercentile(arr, [lower, upper])

    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        return None, None

    return float(vmin), float(vmax)


def get_coord_if_available(ds, coord_name):
    """
    Try to get a lon/lat coordinate or variable as 1D or 2D.
    """
    if coord_name not in ds:
        return None

    da = ds[coord_name]

    if "file" in da.dims:
        da = da.isel(file=0)

    da = da.squeeze(drop=True)

    if da.ndim not in [1, 2]:
        return None

    arr = da.values

    if not np.isfinite(arr).any():
        return None

    return arr


def get_pcolormesh_coords(ds, data_da):
    """
    Prefer lon/lat if available.
    Otherwise use native array index coordinates.

    Still no transpose and no flip.
    """
    coord_pairs = [
        ("longitude", "latitude"),
        ("lon", "lat"),
        ("LONGITUDE", "LATITUDE"),
    ]

    for lon_name, lat_name in coord_pairs:
        lon = get_coord_if_available(ds, lon_name)
        lat = get_coord_if_available(ds, lat_name)

        if lon is not None and lat is not None:
            return lon, lat, lon_name, lat_name

    ny, nx = data_da.shape

    x = np.arange(nx)
    y = np.arange(ny)

    return x, y, "x_index", "y_index"


def plot_with_mask_pcolormesh(ax, x, y, data, mask, title, cmap="magma"):
    """
    Plot data with pcolormesh and overlay mask with pcolormesh.
    """
    data = np.asarray(data, dtype=np.float32)
    mask = np.asarray(mask, dtype=np.uint8)

    vmin, vmax = robust_limits(data)

    mesh = ax.pcolormesh(
        x,
        y,
        data,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        shading="auto",
    )

    mask_overlay = np.ma.masked_where(mask == 0, mask)

    ax.pcolormesh(
        x,
        y,
        mask_overlay,
        cmap="Reds",
        alpha=0.30,
        shading="auto",
    )

    if np.any(mask == 1):
        try:
            ax.contour(
                x,
                y,
                mask,
                levels=[0.5],
                colors="cyan",
                linewidths=1.5,
            )
        except Exception as e:
            print(f"Could not draw contour outline: {e}")

    ax.set_title(title)
    ax.set_xlabel("x / longitude")
    ax.set_ylabel("y / latitude")

    return mesh


def plot_random_processed_nc(seed=None):
    """
    Pick a random processed NetCDF file and plot:
      1. original SSHA + bbox mask
      2. rolling highpass + bbox mask
    """
    if seed is not None:
        random.seed(seed)

    nc_files = sorted(PROCESSED_DIR.glob("*.nc"))

    if not nc_files:
        raise FileNotFoundError(f"No .nc files found in {PROCESSED_DIR}")

    nc_file = random.choice(nc_files)

    print("Random file selected:")
    print(nc_file)

    with xr.open_dataset(nc_file) as ds:
        ssha_da = get_2d_da(ds, BASE_VAR)
        highpass_da = get_2d_da(ds, HIGHPASS_VAR)
        mask_da = get_2d_da(ds, MASK_VAR)

        ssha = ssha_da.values
        highpass = highpass_da.values
        mask = mask_da.values.astype(np.uint8)

        x, y, x_name, y_name = get_pcolormesh_coords(ds, ssha_da)

    if ssha.shape != highpass.shape:
        raise ValueError(
            f"Shape mismatch: {BASE_VAR} shape={ssha.shape}, "
            f"{HIGHPASS_VAR} shape={highpass.shape}"
        )

    if ssha.shape != mask.shape:
        raise ValueError(
            f"Shape mismatch: {BASE_VAR} shape={ssha.shape}, "
            f"{MASK_VAR} shape={mask.shape}"
        )

    print(f"SSHA shape:       {ssha.shape}")
    print(f"Highpass shape:   {highpass.shape}")
    print(f"Mask shape:       {mask.shape}")
    print(f"Mask pixels:      {int(np.sum(mask == 1))}")
    print(f"pcolormesh x:     {x_name}, shape={np.shape(x)}")
    print(f"pcolormesh y:     {y_name}, shape={np.shape(y)}")

    fig, axes = plt.subplots(
        nrows=1,
        ncols=2,
        figsize=(16, 7),
        constrained_layout=True,
    )

    im0 = plot_with_mask_pcolormesh(
        axes[0],
        x,
        y,
        ssha,
        mask,
        title=f"Original SSHA + bbox mask\n{nc_file.name}",
    )

    im1 = plot_with_mask_pcolormesh(
        axes[1],
        x,
        y,
        highpass,
        mask,
        title=f"Rolling high-pass + bbox mask\n{nc_file.name}",
    )

    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04, label=BASE_VAR)
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04, label=HIGHPASS_VAR)

    plt.show()

    return nc_file


# Run once
selected_file = plot_random_processed_nc()